# GSEA

In [1]:
# Install GSEApy if you haven't already
# !pip install gseapy

import pandas as pd
import gseapy as gp

def prepare_ranking(csv_file):
    # Load the CSV file containing Ensembl names, symbols, and ranks
    #csv_file = "/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/Results/feature_selection/RNAseq/ridge_L2_0/Experiment_GSE60590_feature_selection_Symbols.csv"
    gene_rank_df = pd.read_csv(csv_file)
    # check the first rows of the dataframe, if it starts with "","coef","abs_coef","NULL", replace NULL with "Symbol" and "" with "Ensembl"
    gene_rank_df.columns = ["Ensembl","coef","abs_coef","Symbol"]
    if gene_rank_df.columns[0] == "V1":
        gene_rank_df.columns = ["Ensembl","coef","abs_coef","Symbol"]
        # remove first row
        gene_rank_df = gene_rank_df.iloc[1:]
    # Add a column rank form 1 to len(gene_rank_df) with the rank of the gene depending on the abs_coef
    gene_rank_df['rank'] = gene_rank_df['abs_coef'].rank(ascending=False)
    return gene_rank_df



In [4]:
def get_GSEA(rank_dict):
    rank_file_path = "rank_file.txt"

    with open(rank_file_path, 'w') as file:
        for gene, rank in rank_dict.items():
            file.write(f"{gene}\t{rank}\n")


    results = gp.prerank(rnk=rank_file_path, gene_sets='Aging_Perturbations_from_GEO_down', outdir='gsea_results', min_size=10, max_size=1000, permutation_num=1000)
    return results

def main(file_name, csv_path = "Results/feature_selection/RNAseq/ridge_L2_0/"):
    
    #file_name = "Experiment_GSE60590_feature_selection_Symbols.csv"
    gene_rank_df = prepare_ranking(csv_path+file_name)
    rank_dict = dict(zip(gene_rank_df["Symbol"], gene_rank_df["rank"]))
    results = get_GSEA(rank_dict)
    enrichments = results.res2d
    best_enrichments = enrichments[enrichments['FDR q-val']<0.1].sort_values('FDR q-val', ascending=True)
    # save the results to a csv file
    best_enrichments.to_csv(f"{csv_path}GSEA/{file_name[:-4]}Aging_Perturbations_from_GEO_down.csv")


In [24]:
# for file in os.listdir("Results/feature_selection/RNAseq/ridge_L2_0/"), run the main function
import os
csv_path = "/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/Results/feature_selection/RNAseq/"
file_list = [#"feature_importance_smote_catboost_7b_top_genes_elbow_mapp.csv", 
             "feature_importance_smote_ridge_7b_top_genes_elbow_mapp.csv"]
for file in file_list:
        main(file, csv_path)
#main("Experiment_GSE60590_feature_selection_Symbols.csv")

2024-09-27 17:12:54,329 [WARNING] Input gene rankings contains NA values(gene name and ranking value), drop them all!


In [21]:
# Load the CSV file containing Ensembl names, symbols, and ranks
csv_file = "/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/Results/feature_selection/RNAseq/ridge_L2_0/Status_Sarcopenia_feature_selection_Symbols.csv"
csv_file = csv_path+file_list[1]
gene_rank_df = prepare_ranking(csv_file)
gene_rank_df

,Ensembl,coef,abs_coef,Symbol,rank
0,ENSG00000004848.7,1.272214,1.272214,ARX,138.0
1,ENSG00000009307.15,1.614391,1.614391,CSDE1,104.0
2,ENSG00000015479.18,2.187071,2.187071,MATR3,63.0
3,ENSG00000022267.16,2.043561,2.043561,FHL1,69.0
4,ENSG00000036448.9,0.849731,0.849731,MYOM2,240.0
...,...,...,...,...,...
340,ENSG00000273079.5,-0.677420,0.677420,GRIN2B,329.0
341,ENSG00000278395.1,-0.754326,0.754326,NaN,286.0
342,ENSG00000280646.2,-0.815350,0.815350,RNA5SP196,251.0
343,ENSG00000281420.1,-0.763942,0.763942,NaN,281.0


In [8]:
csv_file

'/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/Results/feature_selection/RNAseq/feature_importance_smote_catboost_7b_top_genes_elbow_mapp.csv'

In [22]:

# Create a rank dictionary with Ensembl names as keys and ranks as values
rank_dict = dict(zip(gene_rank_df['Symbol'], gene_rank_df['rank']))

# get the symbos with abs_coef > 0.1
gene_rank_df_f = gene_rank_df[gene_rank_df['abs_coef'] > 0.5]
symbols = gene_rank_df_f['Symbol'].tolist()
#remove nan values
symbols = [x for x in symbols if str(x) != 'nan']
gp.Enrichr(gene_list=symbols, gene_sets='KEGG_2019_Human', outdir='test/enrichr_kegg',cutoff=0.1)

In [17]:
symbols

['NT5C2',
 'UBFD1',
 'LOC112694756',
 'H3-3B',
 'MTFR1',
 'BLCAP',
 'FEZ2',
 'PTP4A3',
 'MICAL3',
 'ZNF710',
 'SLC16A3',
 'STUM']

In [10]:
gene_rank_df

,Ensembl,coef,abs_coef,Symbol,rank
0,ENSG00000076685.18,14.135628,14.135628,NT5C2,1.0
1,ENSG00000262855.1,12.041862,12.041862,NaN,2.0
2,ENSG00000103353.15,10.838371,10.838371,UBFD1,3.0
3,ENSG00000285043.1,7.641741,7.641741,LOC112694756,4.0
4,ENSG00000132475.10,6.241443,6.241443,H3-3B,5.0
5,ENSG00000066855.15,4.300535,4.300535,MTFR1,6.0
6,ENSG00000234441.1,4.168808,4.168808,NaN,7.0
7,ENSG00000166619.12,3.897527,3.897527,BLCAP,8.0
8,ENSG00000171055.14,2.825766,2.825766,FEZ2,9.0
9,ENSG00000184489.11,1.917136,1.917136,PTP4A3,10.0


2024-04-02 17:14:30,811 [WARNING] Input gene rankings contains NA values(gene name and ranking value), drop them all!
2024-04-02 17:14:30,817 [WARNING] Duplicated values found in preranked stats: 16.61% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


,Name,Term,ES,NES,NOM p-val,FDR q-val,FWER p-val,Tag %,Gene %,Lead_genes
0,prerank,Selenocompound metabolism,-0.487261,-2.104585,0.0,0.0,0.0,16/16,51.31%,INMT;PAPSS2;TXNRD1;SCLY;KYAT3;TXNRD2;MTR;TXNRD...
1,prerank,Olfactory transduction,0.581038,2.008736,0.0,0.0,0.0,344/434,30.70%,OR4E2;OR8A1;OR4E1;OR2H2;OR10AC1;OR2B2;OR1L6;OR...
2,prerank,Protein export,-0.45193,-2.000342,0.0,0.0,0.0,22/22,54.85%,SEC61G;SEC11A;IMMP1L;SRP68;SEC61A2;SPCS2;SRP72...
7,prerank,Maturity onset diabetes of the young,0.558944,1.599368,0.003012,0.062392,0.166,21/25,38.71%,HNF1A;NKX6-1;PKLR;FOXA2;INS;NEUROG3;SLC2A2;RFX...
9,prerank,Taste transduction,0.486326,1.544753,0.0,0.086176,0.318,56/75,38.73%,TRPM5;ASIC2;GABRA5;TAS2R46;TAS2R40;HTR3B;TAS2R...
8,prerank,Fructose and mannose metabolism,-0.376116,-1.598311,0.0,0.146939,0.012407,30/30,62.44%,HK2;PFKFB2;TPI1;TKFC;ALDOA;PFKM;PFKFB1;HK1;TIG...
5,prerank,Terpenoid backbone biosynthesis,-0.405272,-1.69386,0.0,0.164571,0.009926,19/19,59.51%,GGPS1;FDPS;HMGCS2;HMGCS1;PDSS2;MVK;ACAT1;DHDDS...
6,prerank,Fatty acid biosynthesis,-0.457167,-1.625537,0.0,0.171429,0.012407,13/13,54.31%,ACSL3;ACSL1;ACACB;MCAT;ACSL5;OXSM;FASN;ACSL6;A...
3,prerank,Glycosylphosphatidylinositol (GPI)-anchor bios...,-0.43626,-1.744294,0.0,0.205714,0.007444,25/25,56.42%,PIGS;PIGF;PIGU;PIGP;PIGL;PIGX;PIGT;DPM2;PIGB;P...
4,prerank,SNARE interactions in vesicular transport,-0.342579,-1.701688,0.0,0.205714,0.009926,33/33,65.79%,SNAP29;GOSR2;VTI1B;VAMP2;STX8;BET1;GOSR1;VAMP3...


In [15]:
#!pip install lxml
from gseapy import Msigdb
msig = Msigdb()
# mouse hallmark gene sets
# get database list
#msig.gene_sets

/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/.venv/lib/python3.10/site-packages/gseapy/msigdb.py:20: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  d = pd.read_html(resp.text)[0]
/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/.venv/lib/python3.10/site-packages/gseapy/msigdb.py:72: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  d = pd.read_html(resp.text)[0]


AttributeError: 'Msigdb' object has no attribute 'gene_sets'

In [8]:
import gseapy as gp
databases = gp.get_library_name()
# save on txt file
with open("Data/GSEA_Avaliable_databases.txt", "w") as file:
    for db in databases:
        file.write(f"{db}\n")

In [ ]:

# Print top enriched gene sets
print(results.res2d.head())

# Visualize enrichment results
gp.plot.gsea_plot(rank_metric=results.ranking, term=results.res2d.index[0], **results.results[results.res2d.index[0]])
